# CallMeMaybe — Operational Service Performance Analysis

## Professional portfolio rebuild

This analysis evaluates operator-assigned call activity through a transparent operational KPI framework. It replaces the original definitive *inefficient operator* classification with system-level reporting, operator review signals, volume context, missed-call concentration, time trends, and separate outbound activity analysis.

### Analytical scope

- Primary operator-review population: cleaned records with an assigned `operator_id`.
- Calls are measured with `calls_count`; database records are not treated as individual calls.
- Waiting time is calculated per represented call.
- Records without an operator are retained for data-scope validation but are not attributed to operators.
- The 10% missed-rate line is a **project analytical reference**, not an SLA or external benchmark.
- Inferential tests from the original TripleTen submission are preserved only in the protected academic-history file and are not part of this professional methodology.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CALLS_FILE = DATA_DIR / "telecom_dataset_us.csv"
CLIENTS_FILE = DATA_DIR / "telecom_clients_us.csv"
PROJECT_MISSED_RATE_REFERENCE = 0.10
EXPLORATORY_VOLUME_VIEW = 30

## 1. Source validation and data-quality reconciliation

Exact duplicates are removed across the original nine fields. Missing `operator_id` and `internal` values are not imputed. The source-reported calendar date is retained so timezone conversion does not shift reporting dates backward.

In [ ]:
calls_raw = pd.read_csv(CALLS_FILE)
clients = pd.read_csv(CLIENTS_FILE)

expected_call_fields = [
    "user_id", "date", "direction", "internal", "operator_id",
    "is_missed_call", "calls_count", "call_duration", "total_call_duration"
]
expected_client_fields = ["user_id", "tariff_plan", "date_start"]

assert calls_raw.shape == (53_902, 9)
assert calls_raw.columns.tolist() == expected_call_fields
assert clients.shape == (732, 3)
assert clients.columns.tolist() == expected_client_fields
assert calls_raw.duplicated().sum() == 4_900

calls = calls_raw.drop_duplicates().copy()
calls["report_date"] = pd.to_datetime(calls["date"].str.slice(0, 10), format="%Y-%m-%d")
calls["week_start"] = calls["report_date"] - pd.to_timedelta(calls["report_date"].dt.weekday, unit="D")
calls["wait_total_seconds"] = calls["total_call_duration"] - calls["call_duration"]
calls["call_type"] = calls["internal"].map({True: "Internal", False: "External"}).fillna("Unknown")

assert calls.shape[0] == 49_002
assert (calls["calls_count"] > 0).all()
assert (calls["wait_total_seconds"] >= 0).all()
assert set(calls["direction"].unique()) == {"in", "out"}

assigned = calls[calls["operator_id"].notna()].copy()
assigned["operator_id"] = assigned["operator_id"].astype("int64")
unassigned = calls[calls["operator_id"].isna()].copy()
inbound = assigned[assigned["direction"].eq("in")].copy()
outbound = assigned[assigned["direction"].eq("out")].copy()

scope_validation = pd.DataFrame([
    ["Source records", len(calls_raw), "Source fingerprint"],
    ["Exact duplicates removed", calls_raw.duplicated().sum(), "Exact full-row duplicates"],
    ["Cleaned records", len(calls), "Primary cleaned dataset"],
    ["Operator-assigned records", len(assigned), "Primary operator-review scope"],
    ["Records without operator_id", len(unassigned), "Operational meaning requires verification"],
    ["Unique assigned operators", assigned["operator_id"].nunique(), "All directions"],
    ["Operators with inbound activity", inbound["operator_id"].nunique(), "Inbound operator scope"],
    ["Operators with outbound activity", outbound["operator_id"].nunique(), "Outbound descriptive scope"],
], columns=["measure", "value", "scope_note"])

scope_validation.to_csv(OUTPUT_DIR / "data_scope_validation.csv", index=False)
scope_validation

### Records without an assigned operator

The following calculation is retained only as **DESCRIPTIVE DATA-SCOPE RESULT — OPERATIONAL MEANING REQUIRES VERIFICATION**. It is not an executive KPI or operator-performance result.

In [ ]:
all_inbound = calls[calls["direction"].eq("in")]
all_outbound = calls[calls["direction"].eq("out")]
unassigned_inbound = unassigned[unassigned["direction"].eq("in")]

unassigned_scope = pd.DataFrame({
    "measure": [
        "Inbound calls without operator_id",
        "Missed inbound calls without operator_id",
        "All-cleaned-record inbound attempts",
        "All-cleaned-record missed inbound calls",
        "All-cleaned-record missed rate",
    ],
    "value": [
        unassigned_inbound["calls_count"].sum(),
        unassigned_inbound.loc[unassigned_inbound["is_missed_call"], "calls_count"].sum(),
        all_inbound["calls_count"].sum(),
        all_inbound.loc[all_inbound["is_missed_call"], "calls_count"].sum(),
        all_inbound.loc[all_inbound["is_missed_call"], "calls_count"].sum() / all_inbound["calls_count"].sum(),
    ],
    "classification": "DESCRIPTIVE DATA-SCOPE RESULT — OPERATIONAL MEANING REQUIRES VERIFICATION",
})
unassigned_scope.to_csv(OUTPUT_DIR / "missing_operator_scope_validation.csv", index=False)
unassigned_scope

## 2. Operator-assigned system KPIs

These KPIs apply specifically to the operator-assigned population. They must not be described as unrestricted call-centre results.

In [ ]:
inbound_attempts = inbound["calls_count"].sum()
missed_inbound_calls = inbound.loc[inbound["is_missed_call"], "calls_count"].sum()
total_inbound_wait = inbound["wait_total_seconds"].sum()
outbound_calls = outbound["calls_count"].sum()
total_assigned_calls = inbound_attempts + outbound_calls

system_kpis = pd.DataFrame({
    "kpi": [
        "Operator-assigned inbound attempts",
        "Operator-assigned missed inbound calls",
        "Operator-assigned missed inbound rate",
        "Operator-assigned total inbound waiting time (seconds)",
        "Operator-assigned average inbound wait per call (seconds)",
        "Operator-assigned outbound calls",
        "Inbound share of operator-assigned call volume",
        "Outbound share of operator-assigned call volume",
    ],
    "value": [
        inbound_attempts,
        missed_inbound_calls,
        missed_inbound_calls / inbound_attempts,
        total_inbound_wait,
        total_inbound_wait / inbound_attempts,
        outbound_calls,
        inbound_attempts / total_assigned_calls,
        outbound_calls / total_assigned_calls,
    ],
    "population": "Cleaned records with assigned operator_id",
})

expected_values = {
    "Operator-assigned inbound attempts": 93_802,
    "Operator-assigned missed inbound calls": 926,
    "Operator-assigned missed inbound rate": 926 / 93_802,
    "Operator-assigned total inbound waiting time (seconds)": 1_233_285,
    "Operator-assigned average inbound wait per call (seconds)": 1_233_285 / 93_802,
    "Operator-assigned outbound calls": 608_343,
}
for kpi, expected in expected_values.items():
    actual = system_kpis.loc[system_kpis["kpi"].eq(kpi), "value"].iloc[0]
    assert np.isclose(actual, expected)

system_kpis.to_csv(OUTPUT_DIR / "system_kpi_validation.csv", index=False)
system_kpis

## 3. Corrected operator-level analytical dataset

Volume is displayed alongside every rate. The 10% field is a project reference only, and the 30-call field is an exploratory sensitivity view—not an eligibility rule.

In [ ]:
operator_inbound = inbound.groupby("operator_id").agg(
    inbound_attempts=("calls_count", "sum"),
    inbound_records=("calls_count", "size"),
    total_inbound_wait_seconds=("wait_total_seconds", "sum"),
    active_inbound_days=("report_date", "nunique"),
).reset_index()

operator_missed = (
    inbound[inbound["is_missed_call"]]
    .groupby("operator_id", as_index=False)["calls_count"].sum()
    .rename(columns={"calls_count": "missed_inbound_calls"})
)
operator_inbound = operator_inbound.merge(operator_missed, on="operator_id", how="left")
operator_inbound["missed_inbound_calls"] = operator_inbound["missed_inbound_calls"].fillna(0).astype(int)
operator_inbound["missed_inbound_rate"] = operator_inbound["missed_inbound_calls"] / operator_inbound["inbound_attempts"]
operator_inbound["avg_inbound_wait_per_call_seconds"] = operator_inbound["total_inbound_wait_seconds"] / operator_inbound["inbound_attempts"]
operator_inbound["share_total_missed_calls"] = operator_inbound["missed_inbound_calls"] / missed_inbound_calls
operator_inbound["active_inbound_weeks"] = inbound.groupby("operator_id")["week_start"].nunique().reindex(operator_inbound["operator_id"]).to_numpy()
operator_inbound["above_10pct_project_reference"] = operator_inbound["missed_inbound_rate"] > PROJECT_MISSED_RATE_REFERENCE
operator_inbound["exploratory_30plus_calls_view"] = operator_inbound["inbound_attempts"] >= EXPLORATORY_VOLUME_VIEW

operator_outbound = outbound.groupby("operator_id").agg(
    outbound_calls=("calls_count", "sum"),
    outbound_records=("calls_count", "size"),
    active_outbound_days=("report_date", "nunique"),
    active_outbound_weeks=("week_start", "nunique"),
).reset_index()
operator_outbound["outbound_calls_per_active_day"] = operator_outbound["outbound_calls"] / operator_outbound["active_outbound_days"]

operator_kpis = operator_inbound.merge(operator_outbound, on="operator_id", how="outer")
for column in ["inbound_attempts", "missed_inbound_calls", "outbound_calls"]:
    operator_kpis[column] = operator_kpis[column].fillna(0)
operator_kpis["total_assigned_calls"] = operator_kpis["inbound_attempts"] + operator_kpis["outbound_calls"]
operator_kpis["inbound_call_share"] = operator_kpis["inbound_attempts"] / operator_kpis["total_assigned_calls"]
operator_kpis["outbound_call_share"] = operator_kpis["outbound_calls"] / operator_kpis["total_assigned_calls"]

assert len(operator_kpis) == 1_092
assert operator_kpis["inbound_attempts"].sum() == 93_802
assert operator_kpis["missed_inbound_calls"].sum() == 926
assert operator_kpis["outbound_calls"].sum() == 608_343
assert operator_kpis["above_10pct_project_reference"].fillna(False).sum() == 33
assert (operator_kpis["above_10pct_project_reference"].fillna(False) & operator_kpis["exploratory_30plus_calls_view"].fillna(False)).sum() == 5
assert operator_kpis["avg_inbound_wait_per_call_seconds"].max() == 115.5

operator_kpis.to_csv(OUTPUT_DIR / "operator_kpis_tableau.csv", index=False)
operator_kpis.head()

## 4. Rate, volume, and missed-call contribution

The scatterplot is a review aid, not an employee classification. The horizontal line is a project analytical reference.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_data = operator_inbound.copy()
sizes = 25 + 10 * np.sqrt(plot_data["missed_inbound_calls"])
scatter = ax.scatter(
    plot_data["inbound_attempts"], plot_data["missed_inbound_rate"] * 100,
    s=sizes, c=plot_data["missed_inbound_calls"], cmap="viridis", alpha=0.72
)
ax.axhline(PROJECT_MISSED_RATE_REFERENCE * 100, color="#C44E52", linestyle="--",
           label="10% project analytical reference")
ax.set_xscale("log")
ax.set_xlabel("Inbound attempts (log scale)")
ax.set_ylabel("Missed inbound rate (%)")
ax.set_title("Operator missed-call rate with inbound volume context")
ax.legend()
fig.colorbar(scatter, ax=ax, label="Missed inbound calls")
plt.tight_layout()
plt.show()

sensitivity = pd.DataFrame({
    "measure": [
        "Operators above 10% project reference — all inbound volumes",
        "Median inbound calls among operators above 10%",
        "Operators in exploratory 30+ inbound-call view",
        "Operators above 10% within exploratory 30+ call view",
        "Missed calls contributed by operators above 10%",
        "Share of missed calls contributed by operators above 10%",
    ],
    "value": [
        operator_inbound["above_10pct_project_reference"].sum(),
        operator_inbound.loc[operator_inbound["above_10pct_project_reference"], "inbound_attempts"].median(),
        operator_inbound["exploratory_30plus_calls_view"].sum(),
        (operator_inbound["above_10pct_project_reference"] & operator_inbound["exploratory_30plus_calls_view"]).sum(),
        operator_inbound.loc[operator_inbound["above_10pct_project_reference"], "missed_inbound_calls"].sum(),
        operator_inbound.loc[operator_inbound["above_10pct_project_reference"], "missed_inbound_calls"].sum() / missed_inbound_calls,
    ],
    "classification": [
        "PROJECT ANALYTICAL ASSUMPTION RESULT",
        "DESCRIPTIVE OBSERVATION",
        "SENSITIVITY POPULATION",
        "SENSITIVITY RESULT",
        "DESCRIPTIVE OBSERVATION",
        "DESCRIPTIVE OBSERVATION",
    ],
})
sensitivity.to_csv(OUTPUT_DIR / "operator_sensitivity_results.csv", index=False)
sensitivity

## 5. Missed-call concentration and Pareto analysis

In [ ]:
pareto = operator_inbound.sort_values(
    ["missed_inbound_calls", "inbound_attempts"], ascending=False
).copy()
pareto["operator_rank_by_missed_calls"] = np.arange(1, len(pareto) + 1)
pareto["cumulative_missed_calls"] = pareto["missed_inbound_calls"].cumsum()
pareto["cumulative_missed_share"] = pareto["cumulative_missed_calls"] / missed_inbound_calls
pareto.to_csv(OUTPUT_DIR / "operator_missed_call_pareto_tableau.csv", index=False)

positive_pareto = pareto[pareto["missed_inbound_calls"] > 0]
operators_for_50 = int((positive_pareto["cumulative_missed_share"] < 0.50).sum() + 1)
operators_for_80 = int((positive_pareto["cumulative_missed_share"] < 0.80).sum() + 1)
assert operators_for_50 == 27
assert operators_for_80 == 92

pareto_summary = pd.DataFrame({
    "measure": [
        "Operators with at least one missed inbound call",
        "Operators accounting for 50% of missed calls",
        "Operators accounting for 80% of missed calls",
        "Top 5 operator share",
        "Top 10 operator share",
        "Top 20 operator share",
    ],
    "value": [
        len(positive_pareto), operators_for_50, operators_for_80,
        positive_pareto.head(5)["missed_inbound_calls"].sum() / missed_inbound_calls,
        positive_pareto.head(10)["missed_inbound_calls"].sum() / missed_inbound_calls,
        positive_pareto.head(20)["missed_inbound_calls"].sum() / missed_inbound_calls,
    ],
})
pareto_summary.to_csv(OUTPUT_DIR / "pareto_validation.csv", index=False)

fig, ax1 = plt.subplots(figsize=(11, 6))
visible = positive_pareto.head(120)
ax1.bar(visible["operator_rank_by_missed_calls"], visible["missed_inbound_calls"], color="#4C72B0")
ax1.set_xlabel("Operator rank by missed inbound calls")
ax1.set_ylabel("Missed inbound calls", color="#4C72B0")
ax2 = ax1.twinx()
ax2.plot(visible["operator_rank_by_missed_calls"], visible["cumulative_missed_share"] * 100, color="#C44E52")
ax2.axhline(80, color="#C44E52", linestyle="--", alpha=0.6)
ax2.set_ylabel("Cumulative share of missed calls (%)", color="#C44E52")
ax1.set_title("Missed inbound call concentration by operator")
plt.tight_layout()
plt.show()

pareto_summary

## 6. Corrected time-based analysis

Weeks begin Monday. Boundary weeks are retained and labelled partial but are excluded from comparative extrema.

In [ ]:
system_week = inbound.groupby("week_start").agg(
    weekly_inbound_attempts=("calls_count", "sum"),
    weekly_total_wait_seconds=("wait_total_seconds", "sum"),
    weekly_active_operators=("operator_id", "nunique"),
).reset_index()
weekly_missed = (
    inbound[inbound["is_missed_call"]]
    .groupby("week_start", as_index=False)["calls_count"].sum()
    .rename(columns={"calls_count": "weekly_missed_inbound_calls"})
)
system_week = system_week.merge(weekly_missed, on="week_start", how="left")
system_week["weekly_missed_inbound_calls"] = system_week["weekly_missed_inbound_calls"].fillna(0).astype(int)
system_week["weekly_missed_inbound_rate"] = system_week["weekly_missed_inbound_calls"] / system_week["weekly_inbound_attempts"]
system_week["weekly_avg_inbound_wait_seconds"] = system_week["weekly_total_wait_seconds"] / system_week["weekly_inbound_attempts"]
system_week["is_partial_boundary_week"] = False
system_week.loc[system_week.index[[0, -1]], "is_partial_boundary_week"] = True
system_week.to_csv(OUTPUT_DIR / "system_week_kpis_tableau.csv", index=False)

operator_week = inbound.groupby(["operator_id", "week_start"]).agg(
    weekly_inbound_attempts=("calls_count", "sum"),
    weekly_total_wait_seconds=("wait_total_seconds", "sum"),
    active_inbound_days=("report_date", "nunique"),
).reset_index()
operator_week_missed = (
    inbound[inbound["is_missed_call"]]
    .groupby(["operator_id", "week_start"], as_index=False)["calls_count"].sum()
    .rename(columns={"calls_count": "weekly_missed_inbound_calls"})
)
operator_week = operator_week.merge(operator_week_missed, on=["operator_id", "week_start"], how="left")
operator_week["weekly_missed_inbound_calls"] = operator_week["weekly_missed_inbound_calls"].fillna(0).astype(int)
operator_week["weekly_missed_inbound_rate"] = operator_week["weekly_missed_inbound_calls"] / operator_week["weekly_inbound_attempts"]
operator_week["weekly_avg_inbound_wait_seconds"] = operator_week["weekly_total_wait_seconds"] / operator_week["weekly_inbound_attempts"]
operator_week["is_partial_boundary_week"] = operator_week["week_start"].isin(system_week.loc[system_week["is_partial_boundary_week"], "week_start"])
operator_week.to_csv(OUTPUT_DIR / "operator_week_kpis_tableau.csv", index=False)

assert system_week["weekly_inbound_attempts"].sum() == 93_802
assert system_week["weekly_missed_inbound_calls"].sum() == 926
assert operator_week["weekly_inbound_attempts"].sum() == 93_802
assert operator_week["weekly_missed_inbound_calls"].sum() == 926

complete_weeks = system_week[~system_week["is_partial_boundary_week"]]

fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True)
axes[0].plot(system_week["week_start"], system_week["weekly_inbound_attempts"], marker="o")
axes[0].set_ylabel("Inbound calls")
axes[0].set_title("Operator-assigned inbound service trends")
axes[1].plot(system_week["week_start"], system_week["weekly_missed_inbound_rate"] * 100, marker="o", color="#C44E52")
axes[1].set_ylabel("Missed rate (%)")
axes[2].plot(system_week["week_start"], system_week["weekly_avg_inbound_wait_seconds"], marker="o", color="#55A868")
axes[2].set_ylabel("Average wait (seconds)")
axes[2].set_xlabel("Week starting")
plt.tight_layout()
plt.show()

complete_weeks[[
    "week_start", "weekly_inbound_attempts", "weekly_missed_inbound_calls",
    "weekly_missed_inbound_rate", "weekly_avg_inbound_wait_seconds", "weekly_active_operators"
]]

## 7. Corrected wait-time distribution

No operator exceeds the former 180-second assumption after calculating average wait per represented call. The former threshold is not part of the professional classification logic.

In [ ]:
wait_distribution = operator_inbound["avg_inbound_wait_per_call_seconds"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
).rename("seconds").reset_index().rename(columns={"index": "statistic"})
wait_distribution.to_csv(OUTPUT_DIR / "operator_wait_distribution.csv", index=False)

assert (operator_inbound["avg_inbound_wait_per_call_seconds"] > 180).sum() == 0

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(operator_inbound["avg_inbound_wait_per_call_seconds"], bins=35, ax=ax, color="#55A868")
ax.set_xlabel("Operator average inbound wait per represented call (seconds)")
ax.set_title("Corrected operator wait-time distribution")
plt.tight_layout()
plt.show()

wait_distribution

## 8. Separate outbound activity analysis

Outbound activity is descriptive. The dataset does not establish expected outbound responsibility, schedules, tenure, assigned workload, or working hours.

In [ ]:
outbound_distribution = operator_outbound[[
    "outbound_calls", "active_outbound_days", "outbound_calls_per_active_day"
]].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).T.reset_index().rename(columns={"index": "metric"})
outbound_distribution.to_csv(OUTPUT_DIR / "outbound_activity_validation.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(np.log1p(operator_outbound["outbound_calls"]), bins=35, ax=axes[0], color="#4C72B0")
axes[0].set_xlabel("log(1 + outbound calls)")
axes[0].set_title("Outbound volume distribution")
sns.histplot(np.log1p(operator_outbound["outbound_calls_per_active_day"]), bins=35, ax=axes[1], color="#8172B2")
axes[1].set_xlabel("log(1 + outbound calls per active day)")
axes[1].set_title("Outbound activity normalized by active days")
plt.tight_layout()
plt.show()

outbound_distribution

## 9. Internal, external, and unknown inbound calls

Missing `internal` values remain `Unknown`. Results are descriptive and should be interpreted with call volume.

In [ ]:
call_type_kpis = inbound.groupby("call_type").agg(
    inbound_attempts=("calls_count", "sum"),
    total_inbound_wait_seconds=("wait_total_seconds", "sum"),
    database_records=("calls_count", "size"),
).reset_index()
call_type_missed = (
    inbound[inbound["is_missed_call"]]
    .groupby("call_type", as_index=False)["calls_count"].sum()
    .rename(columns={"calls_count": "missed_inbound_calls"})
)
call_type_kpis = call_type_kpis.merge(call_type_missed, on="call_type", how="left")
call_type_kpis["missed_inbound_calls"] = call_type_kpis["missed_inbound_calls"].fillna(0).astype(int)
call_type_kpis["missed_inbound_rate"] = call_type_kpis["missed_inbound_calls"] / call_type_kpis["inbound_attempts"]
call_type_kpis["avg_inbound_wait_seconds"] = call_type_kpis["total_inbound_wait_seconds"] / call_type_kpis["inbound_attempts"]
call_type_kpis.to_csv(OUTPUT_DIR / "call_type_kpis_tableau.csv", index=False)
call_type_kpis

## 10. Before/after analytical reconciliation

In [ ]:
before_after = pd.DataFrame([
    ["Average inbound wait", "57.6-second unweighted mean of operator record averages", "13.15-second call-weighted operator-assigned system average", "SUPERSEDED"],
    ["Operators above 180-second wait", "39", "0", "SUPERSEDED"],
    ["Old inbound union", "72 operators", "No definitive binary classification", "SUPERSEDED"],
    ["Historical three-condition union", "675 operators", "Removed", "SUPERSEDED"],
    ["Outbound activity", "Low volume below 80% of average treated as inefficiency", "Separate descriptive volume and active-day analysis", "REPLACED"],
    ["Minimum-volume handling", "30 database records labelled as 30 calls", "No permanent eligibility rule; optional 30-call sensitivity view", "REPLACED"],
    ["Direction mix", "59.2% outbound / 40.8% inbound record shares", "86.64% outbound / 13.36% inbound represented-call shares", "CORRECTED"],
    ["Statistical tests", "Academic z-test, Kruskal-Wallis, Mann-Whitney and Holm suite", "Archived as academic history; removed from professional methodology", "ARCHIVED"],
    ["Operator conclusion", "Inefficient/potentially inefficient operator label", "Multi-signal operational review with volume context", "REPLACED"],
], columns=["analytical_output", "before", "professional_rebuild", "status"])
before_after.to_csv(OUTPUT_DIR / "before_after_reconciliation.csv", index=False)
before_after

## 11. Final validation

The following checks ensure that every Tableau-ready export reconciles to the approved Step 5 specification.

In [ ]:
qa_checks = pd.DataFrame([
    ["Source fingerprint 53,902 × 9", calls_raw.shape == (53_902, 9)],
    ["4,900 exact duplicates", calls_raw.duplicated().sum() == 4_900],
    ["49,002 cleaned records", len(calls) == 49_002],
    ["41,546 operator-assigned records", len(assigned) == 41_546],
    ["7,456 records without operator", len(unassigned) == 7_456],
    ["93,802 assigned inbound attempts", inbound_attempts == 93_802],
    ["926 assigned missed inbound calls", missed_inbound_calls == 926],
    ["13.15-second corrected average wait", np.isclose(total_inbound_wait / inbound_attempts, 13.147747382785015)],
    ["608,343 assigned outbound calls", outbound_calls == 608_343],
    ["1,092 operator export rows", len(operator_kpis) == 1_092],
    ["Operator export reconciles inbound", operator_kpis["inbound_attempts"].sum() == inbound_attempts],
    ["Operator export reconciles missed", operator_kpis["missed_inbound_calls"].sum() == missed_inbound_calls],
    ["Operator export reconciles outbound", operator_kpis["outbound_calls"].sum() == outbound_calls],
    ["System-week export reconciles inbound", system_week["weekly_inbound_attempts"].sum() == inbound_attempts],
    ["Operator-week export reconciles inbound", operator_week["weekly_inbound_attempts"].sum() == inbound_attempts],
    ["No corrected operator wait above 180 seconds", (operator_inbound["avg_inbound_wait_per_call_seconds"] > 180).sum() == 0],
    ["27 operators reach 50% missed-call share", operators_for_50 == 27],
    ["92 operators reach 80% missed-call share", operators_for_80 == 92],
], columns=["check", "passed"])
qa_checks.to_csv(OUTPUT_DIR / "implementation_qa_checks.csv", index=False)
assert qa_checks["passed"].all()
qa_checks

## Professional interpretation guardrail

This analysis identifies KPI patterns for further operational review. It does not prove that individual employees were inefficient, establish causation, validate the project reference as an external standard, or control for schedules, workload assignments, routing, tenure, call complexity, or role expectations.